# E_S5 Plan B — hierarchical power-law and cubic-shape tree

This notebook implements the current `scheme_E_S4_S5_planB.drawio` logic for the **strong variant**.

1. Global strength is inherited from E_S4 and evaluated separately for `MIC >= 0.8` and `0.6 <= MIC < 0.8`.
2. `Simple = (|Pearson| >= 0.7) AND (|Spearman| >= 0.7)`.
3. Every Simple case receives one free power-law fit, \(f(x)=a x^b+c\).
4. The fitted exponent is interpreted only when the power-law fit is adequate: \(R^2_{\rm power}\ge 0.75\).
5. An adequate fit is Linear when \(|b-1|\le 0.05\); otherwise the sign of \(ab(b-1)\) gives Concave or Convex.
6. Power-law-inadequate cases receive a cubic-polynomial fit.
7. Threshold detection is retained as an explicit switch, but is disabled until a family-independent discontinuity metric is agreed.
8. For the remaining cubic-fit cases:
   - two roots of \(f'(x)\) inside \(0.2\le u\le0.8\) give **Cubic / two-turning-point**;
   - no interior \(f'\) root plus one \(f''\) sign change gives **S-shaped**;
   - all other patterns are **Other/Uncertain**.

All thresholds and switches are placed at the beginning of the step where they are used.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import OptimizeWarning, curve_fit


def locate_repo_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / 'E').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the repository root containing E/.')


REPO_ROOT = locate_repo_root()
E_DIR = REPO_ROOT / 'E'
S4_DIR = E_DIR / 'output' / 'S4_mic_range_filter'
POINTS_PATH = E_DIR / 'output' / 'S3_snr_mic_sweep' / 'scatter_points.npz'
OUTPUT_DIR = E_DIR / 'output' / 'S5_planB_tree_hierarchy'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COHORTS = {
    'mic_ge_0p8': {
        'label': 'MIC >= 0.8',
        'filename': 'mic_0p8_to_1_full.parquet',
    },
    'mic_0p6_to_0p8': {
        'label': '0.6 <= MIC < 0.8',
        'filename': 'mic_0p6_to_0p8_full.parquet',
    },
}

FAMILY_ORDER = [
    'F01', 'F03', 'F05', 'F07', 'F13', 'F15', 'F18',
    'F19', 'F21', 'F22', 'F23', 'F24', 'F25', 'F26',
]
FAMILY_NAMES = {
    'F01': 'Linear positive',
    'F03': 'Power convex positive',
    'F05': 'Power concave positive',
    'F07': 'Saturation positive',
    'F13': 'S-curve positive',
    'F15': 'Threshold positive',
    'F18': 'U-shape',
    'F19': 'Spike',
    'F21': 'Cubic',
    'F22': 'Oscillation / complex non-monotonic',
    'F23': 'Two Lines',
    'F24': 'Line + Parabola',
    'F25': 'Multi-regime',
    'F26': 'Windowed J',
}
EXPECTED_CLASS = {
    'F01': 'Linear',
    'F03': 'Convex',
    'F05': 'Concave',
    'F07': 'Concave',
    'F13': 'S-shaped',
    'F15': 'Threshold',
    'F18': 'Other/Uncertain',
    'F19': 'Other/Uncertain',
    'F21': 'Cubic',
    'F22': 'Other/Uncertain',
    'F23': 'Other/Uncertain',
    'F24': 'Other/Uncertain',
    'F25': 'Other/Uncertain',
    'F26': 'Other/Uncertain',
}
CLASS_ORDER = [
    'Linear', 'Concave', 'Convex', 'S-shaped',
    'Cubic', 'Threshold', 'Other/Uncertain',
]
COMPACT_FAMILIES = ['F01', 'F03', 'F05', 'F07', 'F13', 'F15', 'F21']

if not POINTS_PATH.exists():
    raise FileNotFoundError(POINTS_PATH)
points = np.load(POINTS_PATH, mmap_mode='r')
print('Scatter points:', POINTS_PATH.relative_to(REPO_ROOT))
print('Output:', OUTPUT_DIR.relative_to(REPO_ROOT))

Scatter points: E/output/S3_snr_mic_sweep/scatter_points.npz
Output: E/output/S5_planB_tree_hierarchy


In [2]:
def power_law(x, a, b, c):
    return c + a * np.power(x, b)


def r2_value(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    rss = float(np.sum((y_true - y_pred) ** 2))
    tss = float(np.sum((y_true - np.mean(y_true)) ** 2))
    if tss <= np.finfo(float).eps:
        return np.nan
    return float(1.0 - rss / tss)


def initial_power_parameters(x, y):
    y_low = float(np.quantile(y, 0.05))
    y_high = float(np.quantile(y, 0.95))
    amplitude = max(y_high - y_low, np.finfo(float).eps)
    covariance = float(np.cov(x, y, ddof=1)[0, 1])
    direction = -1.0 if covariance < 0 else 1.0
    a0 = direction * amplitude
    c0 = y_low if direction > 0 else y_high
    return a0, 1.0, c0


def fit_power_law(x, y, exponent_min, exponent_max):
    result = {
        'converged': False, 'error': '',
        'a': np.nan, 'b': np.nan, 'c': np.nan,
        'r2': np.nan, 'curvature_term': np.nan,
    }
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', OptimizeWarning)
            parameters, _ = curve_fit(
                power_law,
                x,
                y,
                p0=initial_power_parameters(x, y),
                bounds=(
                    [-np.inf, exponent_min, -np.inf],
                    [np.inf, exponent_max, np.inf],
                ),
                maxfev=30000,
            )
        a, b, c = map(float, parameters)
        prediction = power_law(x, a, b, c)
        result.update({
            'converged': True,
            'a': a,
            'b': b,
            'c': c,
            'r2': r2_value(y, prediction),
            'curvature_term': float(a * b * (b - 1.0)),
        })
    except Exception as error:
        result['error'] = f'{type(error).__name__}: {error}'
    return result


def fit_cubic(x, y, interior_lower, interior_upper):
    result = {
        'converged': False, 'error': '',
        'a3': np.nan, 'a2': np.nan, 'a1': np.nan, 'a0': np.nan,
        'r2': np.nan, 'f1_real_root_count': np.nan,
        'f1_interior_root_count': np.nan,
        'f2_root_u': np.nan, 'f2_changes_sign_once': False,
        'direction_sign': 0,
    }
    try:
        order = np.argsort(x)
        x_sorted = np.asarray(x, dtype=float)[order]
        y_sorted = np.asarray(y, dtype=float)[order]
        x_min = float(x_sorted[0])
        x_max = float(x_sorted[-1])
        x_range = x_max - x_min
        if not np.isfinite(x_range) or x_range <= np.finfo(float).eps:
            raise ValueError('Cubic fit requires non-constant x.')

        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            coefficients = np.polyfit(x_sorted, y_sorted, 3)
        polynomial = np.poly1d(coefficients)
        prediction = polynomial(x_sorted)
        a3, a2, a1, a0 = map(float, coefficients)

        derivative_roots = np.roots([3.0 * a3, 2.0 * a2, a1])
        real_roots = np.sort(
            derivative_roots.real[np.abs(derivative_roots.imag) < 1e-8]
        )
        real_root_u = (real_roots - x_min) / x_range
        interior_roots = real_root_u[
            (real_root_u >= interior_lower)
            & (real_root_u <= interior_upper)
        ]

        if abs(a3) > np.finfo(float).eps:
            f2_root = -a2 / (3.0 * a3)
            f2_root_u = float((f2_root - x_min) / x_range)
            f2_changes_once = bool(0.0 < f2_root_u < 1.0)
        else:
            f2_root_u = np.nan
            f2_changes_once = False

        grid = np.linspace(
            x_min + interior_lower * x_range,
            x_min + interior_upper * x_range,
            200,
        )
        first_derivative = np.polyder(polynomial, 1)(grid)
        median_first = float(np.nanmedian(first_derivative))
        direction_sign = 1 if median_first > 0 else (-1 if median_first < 0 else 0)

        result.update({
            'converged': True,
            'a3': a3, 'a2': a2, 'a1': a1, 'a0': a0,
            'r2': r2_value(y_sorted, prediction),
            'f1_real_root_count': int(len(real_roots)),
            'f1_interior_root_count': int(len(interior_roots)),
            'f2_root_u': f2_root_u,
            'f2_changes_sign_once': f2_changes_once,
            'direction_sign': direction_sign,
        })
    except Exception as error:
        result['error'] = f'{type(error).__name__}: {error}'
    return result


def add_feature_rows(frame, positions, rows, prefix):
    if not rows:
        return frame
    feature_frame = pd.DataFrame(rows, index=positions).add_prefix(prefix)
    for column in feature_frame.columns:
        if pd.api.types.is_bool_dtype(feature_frame[column]):
            frame[column] = False
        elif column.endswith('error'):
            frame[column] = ''
        else:
            frame[column] = np.nan
        frame.loc[positions, column] = feature_frame[column].to_numpy()
    return frame


def formatted_count(n, denominator):
    if denominator == 0:
        return '0 (N/A)'
    return f'{int(n)} ({n / denominator:.1%})'


def stage_distribution_table(results, eligible_mask, input_label, outcomes):
    rows = []
    for family_id in FAMILY_ORDER:
        family_mask = results['family_id'].eq(family_id)
        eligible = family_mask & eligible_mask
        denominator = int(eligible.sum())
        if denominator == 0:
            continue
        row = {
            'family_id': family_id,
            'family_name': FAMILY_NAMES[family_id],
            input_label: formatted_count(denominator, denominator),
        }
        for label, mask in outcomes.items():
            row[label] = formatted_count(int((eligible & mask).sum()), denominator)
        rows.append(row)
    return pd.DataFrame(rows)


def final_classification_table(results, family_ids):
    rows = []
    simple = results['simple_relationship']
    for family_id in family_ids:
        family_mask = results['family_id'].eq(family_id) & simple
        denominator = int(family_mask.sum())
        if denominator == 0:
            continue
        expected = EXPECTED_CLASS[family_id]
        row = {
            'family_id': family_id,
            'family_name': FAMILY_NAMES[family_id],
            'Simple-path input': denominator,
        }
        for label in CLASS_ORDER:
            row[label] = formatted_count(
                int((family_mask & results['final_shape'].eq(label)).sum()),
                denominator,
            )
        correct = int((family_mask & results['final_shape'].eq(expected)).sum())
        row['Accuracy'] = f'{correct}/{denominator} ({correct / denominator:.1%})'
        rows.append(row)
    return pd.DataFrame(rows)

## Load the two Strong-variant cohorts

In [3]:
cohort_results = {}
cohort_xy = {}

for cohort_key, cohort_config in COHORTS.items():
    input_path = S4_DIR / cohort_config['filename']
    if not input_path.exists():
        raise FileNotFoundError(input_path)
    frame = pd.read_parquet(input_path)
    frame = frame.loc[
        frame['variant_level'].eq('strong')
    ].sort_values('candidate_index').reset_index(drop=True)
    candidate_indices = frame['candidate_index'].to_numpy(dtype=int)
    x_values = points['x'][candidate_indices].astype(float)
    y_values = points['y'][candidate_indices].astype(float)
    cohort_results[cohort_key] = frame
    cohort_xy[cohort_key] = (x_values, y_values)
    print(cohort_config['label'], 'Strong input:', len(frame))

MIC >= 0.8 Strong input: 7457
0.6 <= MIC < 0.8 Strong input: 2667


## Step 1 — Simple or Complex?

In [4]:
# ============================================================
# STEP 1 CONFIG — SIMPLE / COMPLEX
# ============================================================
SIMPLE_CORRELATION_THRESHOLD = 0.70

for cohort_key, results in cohort_results.items():
    results['simple_relationship'] = (
        results['pearson_r'].abs().ge(SIMPLE_CORRELATION_THRESHOLD)
        & results['spearman_rho'].abs().ge(SIMPLE_CORRELATION_THRESHOLD)
    )

## Step 2 — Fit one power law to every Simple case

In [5]:
# ============================================================
# STEP 2 CONFIG — POWER-LAW FIT AND ADEQUACY
# ============================================================
POWER_EXPONENT_MIN = 0.05
POWER_EXPONENT_MAX = 8.0
POWER_R2_THRESHOLD = 0.75

for cohort_key, results in cohort_results.items():
    x_values, y_values = cohort_xy[cohort_key]
    positions = np.flatnonzero(results['simple_relationship'].to_numpy())
    rows = [
        fit_power_law(
            x_values[position],
            y_values[position],
            POWER_EXPONENT_MIN,
            POWER_EXPONENT_MAX,
        )
        for position in positions
    ]
    results = add_feature_rows(results, positions, rows, 'power_')
    results['power_converged'] = results[
        'power_converged'
    ].fillna(False).astype(bool)
    results['power_adequate'] = (
        results['simple_relationship']
        & results['power_converged']
        & results['power_r2'].ge(POWER_R2_THRESHOLD)
    )
    cohort_results[cohort_key] = results

## Step 3 — Adequate power law: Linear or curvature?

In [6]:
# ============================================================
# STEP 3 CONFIG — LINEARITY AND CURVATURE SIGN
# ============================================================
POWER_EXPONENT_LINEAR_TOLERANCE = 0.05
POWER_CURVATURE_ZERO_TOLERANCE = 1e-10

for cohort_key, results in cohort_results.items():
    results['confirmed_linear'] = (
        results['power_adequate']
        & results['power_b'].sub(1.0).abs().le(
            POWER_EXPONENT_LINEAR_TOLERANCE
        )
    )
    results['curvature_pool'] = (
        results['power_adequate'] & ~results['confirmed_linear']
    )
    curvature = results['power_curvature_term']
    results['power_curvature_shape'] = 'Other/Uncertain'
    results.loc[
        results['curvature_pool']
        & curvature.lt(-POWER_CURVATURE_ZERO_TOLERANCE),
        'power_curvature_shape',
    ] = 'Concave'
    results.loc[
        results['curvature_pool']
        & curvature.gt(POWER_CURVATURE_ZERO_TOLERANCE),
        'power_curvature_shape',
    ] = 'Convex'
    cohort_results[cohort_key] = results

## Step 4 — Power-law inadequate: cubic polynomial fit

In [7]:
# ============================================================
# STEP 4 CONFIG — CUBIC FIT AND INTERIOR ROOT WINDOW
# ============================================================
CUBIC_INTERIOR_LOWER = 0.20
CUBIC_INTERIOR_UPPER = 0.80

for cohort_key, results in cohort_results.items():
    x_values, y_values = cohort_xy[cohort_key]
    results['cubic_pool'] = (
        results['simple_relationship'] & ~results['power_adequate']
    )
    positions = np.flatnonzero(results['cubic_pool'].to_numpy())
    rows = [
        fit_cubic(
            x_values[position],
            y_values[position],
            CUBIC_INTERIOR_LOWER,
            CUBIC_INTERIOR_UPPER,
        )
        for position in positions
    ]
    results = add_feature_rows(results, positions, rows, 'cubic_')
    results['cubic_converged'] = results[
        'cubic_converged'
    ].fillna(False).astype(bool)
    cohort_results[cohort_key] = results

## Step 5 — Threshold evidence, then S-shaped or Cubic

In [8]:
# ============================================================
# STEP 5 CONFIG — THRESHOLD SWITCH AND CUBIC DERIVATIVE PATTERN
# ============================================================
# No family-independent discontinuity detector has been agreed yet.
# Keep this False to avoid using family labels as a hidden shortcut.
ENABLE_THRESHOLD_DETECTOR = False

for cohort_key, results in cohort_results.items():
    results['threshold_detected'] = False
    if ENABLE_THRESHOLD_DETECTOR:
        raise NotImplementedError(
            'Define a family-independent threshold metric before enabling.'
        )

    cubic_supported = (
        results['cubic_pool'] & results['cubic_converged']
    )
    derivative_pool = cubic_supported & ~results['threshold_detected']
    results['cubic_derivative_shape'] = 'Other/Uncertain'

    two_interior_turning_points = (
        derivative_pool
        & results['cubic_f1_interior_root_count'].eq(2)
    )
    monotonic_inflection = (
        derivative_pool
        & results['cubic_f1_interior_root_count'].eq(0)
        & results['cubic_f2_changes_sign_once'].fillna(False).astype(bool)
    )
    results.loc[
        two_interior_turning_points, 'cubic_derivative_shape'
    ] = 'Cubic'
    results.loc[
        monotonic_inflection, 'cubic_derivative_shape'
    ] = 'S-shaped'

    results['final_shape'] = 'Complex / deferred'
    simple = results['simple_relationship']
    results.loc[simple, 'final_shape'] = 'Other/Uncertain'
    results.loc[results['confirmed_linear'], 'final_shape'] = 'Linear'
    results.loc[
        results['curvature_pool']
        & results['power_curvature_shape'].eq('Concave'),
        'final_shape',
    ] = 'Concave'
    results.loc[
        results['curvature_pool']
        & results['power_curvature_shape'].eq('Convex'),
        'final_shape',
    ] = 'Convex'
    results.loc[results['threshold_detected'], 'final_shape'] = 'Threshold'
    results.loc[
        derivative_pool
        & results['cubic_derivative_shape'].eq('S-shaped'),
        'final_shape',
    ] = 'S-shaped'
    results.loc[
        derivative_pool
        & results['cubic_derivative_shape'].eq('Cubic'),
        'final_shape',
    ] = 'Cubic'

    results['direction'] = 'Not applicable'
    results.loc[
        results['confirmed_linear'],
        'direction',
    ] = np.where(
        results.loc[results['confirmed_linear'], 'power_a'].gt(0),
        'Positive',
        'Negative',
    )
    s_mask = results['final_shape'].eq('S-shaped')
    results.loc[s_mask, 'direction'] = np.where(
        results.loc[s_mask, 'cubic_direction_sign'].gt(0),
        'Positive',
        'Negative',
    )
    results['expected_shape'] = results['family_id'].map(EXPECTED_CLASS)
    results['correct'] = (
        results['simple_relationship']
        & results['final_shape'].eq(results['expected_shape'])
    )
    cohort_results[cohort_key] = results

## Stepwise tables and final results

In [9]:
STEP_TITLES = {
    'step1_simple_complex': 'Step 1 — Simple / Complex',
    'step2_power_fit': 'Step 2 — Power-law fit to every Simple case',
    'step3_power_adequacy': 'Step 3 — Power-law R² adequacy',
    'step4_linear': 'Step 4 — Adequate power law: Linear check',
    'step5_curvature': 'Step 5 — Adequate non-linear power law: curvature sign',
    'step6_cubic_fit': 'Step 6 — Power-law inadequate: cubic fit',
    'step7_threshold': 'Step 7 — Threshold evidence',
    'step8_cubic_shape': 'Step 8 — Cubic derivative pattern',
}

all_outputs = {}
for cohort_key, cohort_config in COHORTS.items():
    results = cohort_results[cohort_key]
    all_input = pd.Series(True, index=results.index)
    simple = results['simple_relationship']
    power_fit_input = simple
    power_fitted = simple & results['power_converged']
    adequate = results['power_adequate']
    curvature_pool = results['curvature_pool']
    cubic_pool = results['cubic_pool']
    cubic_supported = cubic_pool & results['cubic_converged']
    derivative_pool = cubic_supported & ~results['threshold_detected']

    step_tables = {
        'step1_simple_complex': stage_distribution_table(
            results, all_input, 'Strong input',
            {
                'Simple': simple,
                'Complex': ~simple,
            },
        ),
        'step2_power_fit': stage_distribution_table(
            results, power_fit_input, 'Simple input',
            {
                'Fit converged': results['power_converged'],
                'Fit failed': ~results['power_converged'],
            },
        ),
        'step3_power_adequacy': stage_distribution_table(
            results, power_fitted, 'Power fit input',
            {
                f'Adequate R2>={POWER_R2_THRESHOLD}': adequate,
                'Inadequate': ~adequate,
            },
        ),
        'step4_linear': stage_distribution_table(
            results, adequate, 'Adequate power-law input',
            {
                'Linear': results['confirmed_linear'],
                'Non-linear': ~results['confirmed_linear'],
            },
        ),
        'step5_curvature': stage_distribution_table(
            results, curvature_pool, 'Curvature input',
            {
                'Concave': results['power_curvature_shape'].eq('Concave'),
                'Convex': results['power_curvature_shape'].eq('Convex'),
                'Other/Uncertain': results['power_curvature_shape'].eq(
                    'Other/Uncertain'
                ),
            },
        ),
        'step6_cubic_fit': stage_distribution_table(
            results, cubic_pool, 'Cubic-fit input',
            {
                'Fit converged': results['cubic_converged'],
                'Fit failed': ~results['cubic_converged'],
            },
        ),
        'step7_threshold': stage_distribution_table(
            results, cubic_supported, 'Cubic-supported input',
            {
                'Threshold': results['threshold_detected'],
                'No threshold evidence': ~results['threshold_detected'],
            },
        ),
        'step8_cubic_shape': stage_distribution_table(
            results, derivative_pool, 'Derivative-pattern input',
            {
                'S-shaped': results['cubic_derivative_shape'].eq('S-shaped'),
                'Cubic': results['cubic_derivative_shape'].eq('Cubic'),
                'Other/Uncertain': results['cubic_derivative_shape'].eq(
                    'Other/Uncertain'
                ),
            },
        ),
    }

    full_table = final_classification_table(results, FAMILY_ORDER)
    compact_table = final_classification_table(results, COMPACT_FAMILIES)
    simple_n = int(simple.sum())
    correct_n = int(results.loc[simple, 'correct'].sum())
    routing = pd.DataFrame([{
        'cohort': cohort_config['label'],
        'strong_input': len(results),
        'simple': simple_n,
        'complex_deferred': int((~simple).sum()),
        'power_adequate': int(adequate.sum()),
        'confirmed_linear': int(results['confirmed_linear'].sum()),
        'curvature_pool': int(curvature_pool.sum()),
        'cubic_pool': int(cubic_pool.sum()),
        'threshold': int(results['threshold_detected'].sum()),
        's_shaped': int(results['final_shape'].eq('S-shaped').sum()),
        'cubic': int(results['final_shape'].eq('Cubic').sum()),
        'other_uncertain': int(
            (simple & results['final_shape'].eq('Other/Uncertain')).sum()
        ),
        'correct_simple': correct_n,
        'simple_path_accuracy': correct_n / simple_n if simple_n else np.nan,
    }])

    cohort_output = OUTPUT_DIR / cohort_key
    cohort_output.mkdir(parents=True, exist_ok=True)
    results.to_parquet(cohort_output / 'case_results.parquet', index=False)
    full_table.to_csv(cohort_output / 'final_table_all_families.csv', index=False)
    compact_table.to_csv(cohort_output / 'final_table_compact.csv', index=False)
    routing.to_csv(cohort_output / 'routing_summary.csv', index=False)
    for step_name, table in step_tables.items():
        table.to_csv(cohort_output / f'{step_name}.csv', index=False)

    all_outputs[cohort_key] = {
        'results': results,
        'step_tables': step_tables,
        'full_table': full_table,
        'compact_table': compact_table,
        'routing': routing,
    }

routing_comparison = pd.concat(
    [output['routing'] for output in all_outputs.values()],
    ignore_index=True,
)
routing_comparison.to_csv(OUTPUT_DIR / 'routing_comparison.csv', index=False)

print('Routing summary')
display(routing_comparison.assign(
    simple_path_accuracy=routing_comparison[
        'simple_path_accuracy'
    ].map('{:.1%}'.format)
))

for cohort_key, cohort_config in COHORTS.items():
    output = all_outputs[cohort_key]
    print(f"\n{'=' * 90}\n{cohort_config['label']} — sequential tables")
    for step_name, table in output['step_tables'].items():
        print(f"\n{STEP_TITLES[step_name]}")
        display(table)
    print(f"\n{cohort_config['label']} — final Simple-path table")
    display(output['compact_table'])

print('Saved:', OUTPUT_DIR.relative_to(REPO_ROOT))

Routing summary


,cohort,strong_input,simple,complex_deferred,power_adequate,confirmed_linear,curvature_pool,cubic_pool,threshold,s_shaped,cubic,other_uncertain,correct_simple,simple_path_accuracy
0,MIC >= 0.8,7457,5091,2366,4333,970,3363,758,0,525,233,0,2966,58.3%
1,0.6 <= MIC < 0.8,2667,910,1757,467,45,422,443,0,234,209,0,763,83.8%



MIC >= 0.8 — sequential tables

Step 1 — Simple / Complex


,family_id,family_name,Strong input,Simple,Complex
0,F01,Linear positive,994 (100.0%),994 (100.0%),0 (0.0%)
1,F03,Power convex positive,806 (100.0%),806 (100.0%),0 (0.0%)
2,F05,Power concave positive,856 (100.0%),856 (100.0%),0 (0.0%)
3,F07,Saturation positive,55 (100.0%),0 (0.0%),55 (100.0%)
4,F13,S-curve positive,1091 (100.0%),1091 (100.0%),0 (0.0%)
5,F15,Threshold positive,1112 (100.0%),1111 (99.9%),1 (0.1%)
6,F18,U-shape,916 (100.0%),0 (0.0%),916 (100.0%)
7,F21,Cubic,549 (100.0%),233 (42.4%),316 (57.6%)
8,F22,Oscillation / complex non-monotonic,987 (100.0%),0 (0.0%),987 (100.0%)
9,F23,Two Lines,71 (100.0%),0 (0.0%),71 (100.0%)



Step 2 — Power-law fit to every Simple case


...


Step 8 — Cubic derivative pattern


,family_id,family_name,Derivative-pattern input,S-shaped,Cubic,Other/Uncertain
0,F01,Linear positive,94 (100.0%),94 (100.0%),0 (0.0%),0 (0.0%)
1,F03,Power convex positive,4 (100.0%),4 (100.0%),0 (0.0%),0 (0.0%)
2,F05,Power concave positive,1 (100.0%),1 (100.0%),0 (0.0%),0 (0.0%)
3,F13,S-curve positive,103 (100.0%),103 (100.0%),0 (0.0%),0 (0.0%)
4,F15,Threshold positive,32 (100.0%),32 (100.0%),0 (0.0%),0 (0.0%)
5,F21,Cubic,209 (100.0%),0 (0.0%),209 (100.0%),0 (0.0%)



0.6 <= MIC < 0.8 — final Simple-path table


,family_id,family_name,Simple-path input,Linear,Concave,Convex,S-shaped,Cubic,Threshold,Other/Uncertain,Accuracy
0,F01,Linear positive,155,45 (29.0%),12 (7.7%),4 (2.6%),94 (60.6%),0 (0.0%),0 (0.0%),0 (0.0%),45/155 (29.0%)
1,F03,Power convex positive,235,0 (0.0%),0 (0.0%),231 (98.3%),4 (1.7%),0 (0.0%),0 (0.0%),0 (0.0%),231/235 (98.3%)
2,F05,Power concave positive,176,0 (0.0%),175 (99.4%),0 (0.0%),1 (0.6%),0 (0.0%),0 (0.0%),0 (0.0%),175/176 (99.4%)
3,F13,S-curve positive,103,0 (0.0%),0 (0.0%),0 (0.0%),103 (100.0%),0 (0.0%),0 (0.0%),0 (0.0%),103/103 (100.0%)
4,F15,Threshold positive,32,0 (0.0%),0 (0.0%),0 (0.0%),32 (100.0%),0 (0.0%),0 (0.0%),0 (0.0%),0/32 (0.0%)
5,F21,Cubic,209,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),209 (100.0%),0 (0.0%),0 (0.0%),209/209 (100.0%)


Saved: E/output/S5_planB_tree_hierarchy
